# Cross-Validation and Hyperparameter Tuning, Explained Simply
Reliable scores, and choosing a model's settings without fooling yourself. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A winery has 178 wines from three grape varieties, with 13 chemical measurements each. We want to predict the variety. The model is a **decision tree**: it learns a series of yes/no questions about the measurements.

### 178 wines
- `load_wine`: a small, famous dataset that comes with scikit-learn: no file needed
- `return_X_y=True, as_frame=True`: give the measurements (X) and the variety (y) as pandas tables
- `y.value_counts()`: how many wines of each variety: 0, 1 and 2

In [ ]:
from sklearn.datasets import load_wine

X, y = load_wine(return_X_y=True, as_frame=True)
print(X.shape)
print(y.value_counts().to_dict())
print(list(X.columns[:5]))

## 2. One split is luck
The test score depends on which wines happen to land in the test set.

### Ten different splits
- `random_state=seed`: a different random split each time
- `max_depth=3`: the tree asks at most 3 questions in a row
- `tree.score`: accuracy on the test wines
- `the output`: the same model scores anywhere from about 87% to 96%, just by luck

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

scores = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.25, random_state=seed)
    tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_tr, y_tr)
    scores.append(round(tree.score(X_te, y_te), 3))
print(scores)
print("lowest:", min(scores), " highest:", max(scores))

## 3. K-fold cross-validation
Split the data into **k** folds. Train on k - 1 folds, test on the remaining one, and rotate so every fold is tested once. Report the average and the spread.

### 5-fold cross-validation
- `cross_val_score(..., cv=5)`: train and test 5 times, each time holding out a different fifth
- `scores.mean()`: the average: a much more reliable score
- `scores.std()`: the standard deviation: how much the five scores vary

In [ ]:
from sklearn.model_selection import cross_val_score

tree = DecisionTreeClassifier(max_depth=3, random_state=0)
scores = cross_val_score(tree, X, y, cv=5)
print("five scores:", scores.round(3))
print(f"average: {scores.mean():.3f}  (spread: {scores.std():.3f})")

## 4. Tuning a hyperparameter
A **hyperparameter** is a setting we choose before training, like max_depth. Choose it by cross-validation on the training set, and keep the test set for the very end.

### Lock away a test set
- `stratify=y`: same mix of varieties in both parts
- `X_test`: not touched again until the final check

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1, stratify=y)
print(len(X_train), "for training and tuning,", len(X_test), "locked away")

### Cross-validation score for each depth
- `for d in depths`: try depths 1 to 8
- `cross_val_score on X_train`: only the training data: the test set stays locked
- `the output`: too shallow is poor; the score levels off after depth 3 or 4

In [ ]:
import matplotlib.pyplot as plt

depths = range(1, 9)
means = []
for d in depths:
    s = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=0),
                        X_train, y_train, cv=5)
    means.append(s.mean())
    print(f"depth {d}: {s.mean():.3f}")
plt.plot(depths, means, marker="o")
plt.xlabel("max_depth"); plt.ylabel("cross-validation accuracy")
plt.show()

## 5. Grid search
**GridSearchCV** tries every combination of settings with cross-validation, and keeps the best.

### Search two settings at once
- `grid`: 5 depths x 4 leaf sizes = 20 combinations
- `min_samples_leaf`: each final answer must be based on at least this many wines
- `GridSearchCV(..., cv=5)`: 20 combinations x 5 folds = 100 trees trained
- `best_params_`: the winning combination

In [ ]:
from sklearn.model_selection import GridSearchCV

grid = {"max_depth": [2, 3, 4, 5, None],
        "min_samples_leaf": [1, 3, 5, 10]}
search = GridSearchCV(DecisionTreeClassifier(random_state=0), grid, cv=5)
search.fit(X_train, y_train)
print("best settings:", search.best_params_)
print("best cross-validation accuracy:", round(search.best_score_, 3))

### The one final test
- `best_estimator_`: the best model, already retrained on all the training data
- `best.score(X_test, ...)`: the first and only time we use the test set

In [ ]:
best = search.best_estimator_
print("test accuracy:", round(best.score(X_test, y_test), 3))

## Try it yourself
1. Use 10 folds instead of 5 (cv=10). Does the average change? What about the spread?
2. Add 'criterion': ['gini', 'entropy'] to the grid. Does the best score improve?
3. Try RandomizedSearchCV with n_iter=10 instead of the full grid.